# 03 — what resists?

**triplet-phase-lock**

This notebook introduces the resistance stage. We align triplet states to reference directions, apply a 45° cosine threshold, compare against a stricter threshold, and study clean, weakly perturbed, strongly perturbed, and noisy trajectories.

Guiding question:

- **Π (resist): what resists?**


## Setup

We continue from the expand and extend stages.

Triplets are defined as:

\\[\nT_k = (N_k, N_{k+1}, N_{k+2})\n\\]\n
Resistance is measured by alignment to a reference direction using cosine similarity:

\\[\n\\cos\\theta_k = \\frac{T_k \\cdot R}{\\|T_k\\|\\,\\|R\\|}\n\\]\n
We compare two thresholds:

\\[\n\\cos\\theta_k \\ge \\frac{1}{\\sqrt{1^2 + 1^2}}\n\\]\n
and a stricter near-phase-lock threshold used to separate clean, weak, strong, and noisy behavior more clearly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True


## Sequence and triplet helpers


In [ ]:
def sequence_n(k):
    k = np.asarray(k, dtype=float)
    return 24.0 * k - 25.0


def sequence_n_perturbed(k, amplitude=8.0, period=6.0):
    k = np.asarray(k, dtype=float)
    return sequence_n(k) + amplitude * np.sin(k / period)


def sequence_n_noisy(k, amplitude=40.0, seed=7):
    k = np.asarray(k, dtype=float)
    rng = np.random.default_rng(seed)
    return sequence_n(k) + rng.normal(loc=0.0, scale=amplitude, size=len(k))


def build_triplets_from_values(vals):
    vals = np.asarray(vals, dtype=float)
    return np.stack([vals[:-2], vals[1:-1], vals[2:]], axis=1)


def normalize_rows(x, eps=1e-12):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, eps)


def cosine_scores(x, reference):
    x_norm = normalize_rows(x)
    r = np.asarray(reference, dtype=float)
    r = r / max(np.linalg.norm(r), 1e-12)
    return x_norm @ r


def resistance_mask_from_scores(scores, threshold):
    return scores >= threshold


def resistance_ratio(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0:
        return 0.0
    return float(mask.mean())


def resistance_margin(scores, threshold):
    return scores - threshold


def local_differences(x):
    if len(x) < 2:
        return np.array([], dtype=float)
    return np.linalg.norm(np.diff(x, axis=0), axis=1)


## Generate clean, weak, strong, and noisy triplets


In [ ]:
num_steps = 80
k = np.arange(1, num_steps + 3)

clean_vals = sequence_n(k)
weak_vals = sequence_n_perturbed(k, amplitude=8.0, period=6.0)
strong_vals = sequence_n_perturbed(k, amplitude=20.0, period=4.0)
noisy_vals = sequence_n_noisy(k, amplitude=40.0, seed=7)

triplets_clean = build_triplets_from_values(clean_vals)
triplets_weak = build_triplets_from_values(weak_vals)
triplets_strong = build_triplets_from_values(strong_vals)
triplets_noisy = build_triplets_from_values(noisy_vals)

print('Triplet shapes:')
print('clean :', triplets_clean.shape)
print('weak  :', triplets_weak.shape)
print('strong:', triplets_strong.shape)
print('noisy :', triplets_noisy.shape)


## Reference directions and thresholds

We compare two reference choices:

- a global balanced direction \(R=(1,1,1)\)
- an empirical clean reference from the mean normalized clean triplet direction

We also compare:

- the 45° threshold
- a stricter near-phase-lock threshold


In [ ]:
reference_global = np.array([1.0, 1.0, 1.0])
reference_clean = normalize_rows(triplets_clean).mean(axis=0)
reference_clean = reference_clean / np.linalg.norm(reference_clean)

threshold_45 = 1.0 / np.sqrt(1.0**2 + 1.0**2)
threshold_strict = 0.9995

print(f'Global reference      : {reference_global}')
print(f'Empirical clean ref   : {np.round(reference_clean, 12)}')
print(f'45° threshold         : {threshold_45:.12f}')
print(f'Strict threshold      : {threshold_strict:.12f}')


## Plot 1 — raw sequence comparison


In [ ]:
plt.figure()
plt.plot(k, clean_vals, marker='o', label='clean')
plt.plot(k, weak_vals, marker='o', label='weak perturbation')
plt.plot(k, strong_vals, marker='o', label='strong perturbation')
plt.plot(k, noisy_vals, marker='o', label='noisy')
plt.xlabel('k')
plt.ylabel('value')
plt.title('Sequence comparison at the resistance stage')
plt.legend()
plt.tight_layout()
plt.show()


## Resistance scores against the global reference

This reproduces the broad global-alignment behavior.


In [ ]:
scores_global_clean = cosine_scores(triplets_clean, reference_global)
scores_global_weak = cosine_scores(triplets_weak, reference_global)
scores_global_strong = cosine_scores(triplets_strong, reference_global)
scores_global_noisy = cosine_scores(triplets_noisy, reference_global)

print(f'Global clean score range : {scores_global_clean.min():.12f} to {scores_global_clean.max():.12f}')
print(f'Global weak score range  : {scores_global_weak.min():.12f} to {scores_global_weak.max():.12f}')
print(f'Global strong score range: {scores_global_strong.min():.12f} to {scores_global_strong.max():.12f}')
print(f'Global noisy score range : {scores_global_noisy.min():.12f} to {scores_global_noisy.max():.12f}')


In [ ]:
score_idx = np.arange(1, len(scores_global_clean) + 1)

plt.figure()
plt.plot(score_idx, scores_global_clean, marker='o', label='clean')
plt.plot(score_idx, scores_global_weak, marker='o', label='weak perturbation')
plt.plot(score_idx, scores_global_strong, marker='o', label='strong perturbation')
plt.plot(score_idx, scores_global_noisy, marker='o', label='noisy')
plt.axhline(threshold_45, linestyle='--', label='45° threshold')
plt.xlabel('k')
plt.ylabel(r'$\cos\theta_k$')
plt.title('Global-reference resistance scores')
plt.legend()
plt.tight_layout()
plt.show()


## Resistance scores against the empirical clean reference

This makes clean / weak / strong / noisy separation more visible by aligning to observed clean behavior rather than only to the broad direction \((1,1,1)\).


In [ ]:
scores_clean = cosine_scores(triplets_clean, reference_clean)
scores_weak = cosine_scores(triplets_weak, reference_clean)
scores_strong = cosine_scores(triplets_strong, reference_clean)
scores_noisy = cosine_scores(triplets_noisy, reference_clean)

print(f'Clean-reference clean score range : {scores_clean.min():.12f} to {scores_clean.max():.12f}')
print(f'Clean-reference weak score range  : {scores_weak.min():.12f} to {scores_weak.max():.12f}')
print(f'Clean-reference strong score range: {scores_strong.min():.12f} to {scores_strong.max():.12f}')
print(f'Clean-reference noisy score range : {scores_noisy.min():.12f} to {scores_noisy.max():.12f}')


In [ ]:
plt.figure()
plt.plot(score_idx, scores_clean, marker='o', label='clean')
plt.plot(score_idx, scores_weak, marker='o', label='weak perturbation')
plt.plot(score_idx, scores_strong, marker='o', label='strong perturbation')
plt.plot(score_idx, scores_noisy, marker='o', label='noisy')
plt.axhline(threshold_45, linestyle='--', label='45° threshold')
plt.axhline(threshold_strict, linestyle='--', label='strict threshold')
plt.xlabel('k')
plt.ylabel(r'$\cos\theta_k$')
plt.title('Clean-reference resistance scores')
plt.legend()
plt.tight_layout()
plt.show()


## Threshold comparison

The 45° threshold checks broad bounded alignment. The stricter threshold reveals more detailed resistance separation.


In [ ]:
mask45_clean = resistance_mask_from_scores(scores_clean, threshold_45)
mask45_weak = resistance_mask_from_scores(scores_weak, threshold_45)
mask45_strong = resistance_mask_from_scores(scores_strong, threshold_45)
mask45_noisy = resistance_mask_from_scores(scores_noisy, threshold_45)

maskS_clean = resistance_mask_from_scores(scores_clean, threshold_strict)
maskS_weak = resistance_mask_from_scores(scores_weak, threshold_strict)
maskS_strong = resistance_mask_from_scores(scores_strong, threshold_strict)
maskS_noisy = resistance_mask_from_scores(scores_noisy, threshold_strict)

print(f'45° resistance ratio — clean : {resistance_ratio(mask45_clean):.6f}')
print(f'45° resistance ratio — weak  : {resistance_ratio(mask45_weak):.6f}')
print(f'45° resistance ratio — strong: {resistance_ratio(mask45_strong):.6f}')
print(f'45° resistance ratio — noisy : {resistance_ratio(mask45_noisy):.6f}')
print()
print(f'Strict resistance ratio — clean : {resistance_ratio(maskS_clean):.6f}')
print(f'Strict resistance ratio — weak  : {resistance_ratio(maskS_weak):.6f}')
print(f'Strict resistance ratio — strong: {resistance_ratio(maskS_strong):.6f}')
print(f'Strict resistance ratio — noisy : {resistance_ratio(maskS_noisy):.6f}')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].plot(score_idx, mask45_clean.astype(float), marker='o', label='clean')
axes[0].plot(score_idx, mask45_weak.astype(float), marker='o', label='weak perturbation')
axes[0].plot(score_idx, mask45_strong.astype(float), marker='o', label='strong perturbation')
axes[0].plot(score_idx, mask45_noisy.astype(float), marker='o', label='noisy')
axes[0].set_ylabel('accepted / rejected')
axes[0].set_title('Resistance masks under the 45° threshold')
axes[0].legend()

axes[1].plot(score_idx, maskS_clean.astype(float), marker='o', label='clean')
axes[1].plot(score_idx, maskS_weak.astype(float), marker='o', label='weak perturbation')
axes[1].plot(score_idx, maskS_strong.astype(float), marker='o', label='strong perturbation')
axes[1].plot(score_idx, maskS_noisy.astype(float), marker='o', label='noisy')
axes[1].set_xlabel('k')
axes[1].set_ylabel('accepted / rejected')
axes[1].set_title('Resistance masks under the strict threshold')
axes[1].legend()

plt.tight_layout()
plt.show()


## Resistance margins

Measure how far above or below threshold each triplet sits:

\\[\nm_k = \\cos\\theta_k - \\tau\n\\]\n
where \(\tau\) is the chosen threshold.


In [ ]:
margin45_clean = resistance_margin(scores_clean, threshold_45)
margin45_weak = resistance_margin(scores_weak, threshold_45)
margin45_strong = resistance_margin(scores_strong, threshold_45)
margin45_noisy = resistance_margin(scores_noisy, threshold_45)

marginS_clean = resistance_margin(scores_clean, threshold_strict)
marginS_weak = resistance_margin(scores_weak, threshold_strict)
marginS_strong = resistance_margin(scores_strong, threshold_strict)
marginS_noisy = resistance_margin(scores_noisy, threshold_strict)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].plot(score_idx, margin45_clean, marker='o', label='clean')
axes[0].plot(score_idx, margin45_weak, marker='o', label='weak perturbation')
axes[0].plot(score_idx, margin45_strong, marker='o', label='strong perturbation')
axes[0].plot(score_idx, margin45_noisy, marker='o', label='noisy')
axes[0].axhline(0.0, linestyle='--', label='threshold boundary')
axes[0].set_ylabel('score minus 45° threshold')
axes[0].set_title('Margins under the 45° threshold')
axes[0].legend()

axes[1].plot(score_idx, marginS_clean, marker='o', label='clean')
axes[1].plot(score_idx, marginS_weak, marker='o', label='weak perturbation')
axes[1].plot(score_idx, marginS_strong, marker='o', label='strong perturbation')
axes[1].plot(score_idx, marginS_noisy, marker='o', label='noisy')
axes[1].axhline(0.0, linestyle='--', label='threshold boundary')
axes[1].set_xlabel('k')
axes[1].set_ylabel('score minus strict threshold')
axes[1].set_title('Margins under the strict threshold')
axes[1].legend()

plt.tight_layout()
plt.show()


## Accepted triplet-path comparison (strict threshold)

Project only strictly accepted triplets into the first two coordinates.


In [ ]:
plt.figure()
plt.plot(triplets_clean[maskS_clean, 0], triplets_clean[maskS_clean, 1], marker='o', label='clean accepted')
plt.plot(triplets_weak[maskS_weak, 0], triplets_weak[maskS_weak, 1], marker='o', label='weak accepted')
plt.plot(triplets_strong[maskS_strong, 0], triplets_strong[maskS_strong, 1], marker='o', label='strong accepted')
plt.plot(triplets_noisy[maskS_noisy, 0], triplets_noisy[maskS_noisy, 1], marker='o', label='noisy accepted')
plt.xlabel(r'$N_k$')
plt.ylabel(r'$N_{k+1}$')
plt.title('Accepted triplet paths under the strict resistance constraint')
plt.legend()
plt.tight_layout()
plt.show()


## Local variation among strictly accepted triplets

Measure how much accepted structure still varies locally after stricter filtering.


In [ ]:
accepted_delta_clean = local_differences(triplets_clean[maskS_clean])
accepted_delta_weak = local_differences(triplets_weak[maskS_weak])
accepted_delta_strong = local_differences(triplets_strong[maskS_strong])
accepted_delta_noisy = local_differences(triplets_noisy[maskS_noisy])

def safe_range(x):
    if len(x) == 0:
        return 'empty'
    return f'{x.min():.12f} to {x.max():.12f}'

print(f'Accepted clean Δ range : {safe_range(accepted_delta_clean)}')
print(f'Accepted weak Δ range  : {safe_range(accepted_delta_weak)}')
print(f'Accepted strong Δ range: {safe_range(accepted_delta_strong)}')
print(f'Accepted noisy Δ range : {safe_range(accepted_delta_noisy)}')


In [ ]:
plt.figure()
if len(accepted_delta_clean):
    plt.plot(np.arange(1, len(accepted_delta_clean) + 1), accepted_delta_clean, marker='o', label='clean accepted Δ')
if len(accepted_delta_weak):
    plt.plot(np.arange(1, len(accepted_delta_weak) + 1), accepted_delta_weak, marker='o', label='weak accepted Δ')
if len(accepted_delta_strong):
    plt.plot(np.arange(1, len(accepted_delta_strong) + 1), accepted_delta_strong, marker='o', label='strong accepted Δ')
if len(accepted_delta_noisy):
    plt.plot(np.arange(1, len(accepted_delta_noisy) + 1), accepted_delta_noisy, marker='o', label='noisy accepted Δ')
plt.xlabel('accepted-step index')
plt.ylabel(r'$\Delta$ among accepted triplets')
plt.title('Local variation after strict resistance filtering')
plt.legend()
plt.tight_layout()
plt.show()


## Summary

This notebook established the **resist** stage:

- defined global and empirical clean reference directions
- computed cosine alignment scores
- applied the 45° threshold and a stricter threshold
- measured resistance masks and ratios
- quantified resistance margins
- visualized accepted triplet paths under stricter filtering

Next notebook:

- compare the full pipeline across stages
- summarize expand → extend → resist
- connect the sequence families in one cross-stage view
